# Imports

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import keras
import numpy as np
import pandas as pd
import transformers
import tokenizers
# import tensorflow as tf
import torch
import tqdm.notebook as tqdm
import sklearn.model_selection
import matplotlib.pyplot as plt

from pathlib import Path
import enum
import gc

In [2]:
keras.__version__, keras.backend.backend()

('3.15.1', 'torch')

In [4]:
torch.cuda.empty_cache()
gc.collect()

26

In [4]:
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         tf.config.experimental.set_memory_growth(gpus[0], True)
#         print(gpus)
#     except:
#         print("No GPU")

gpus = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
if gpus:
    try:
        torch.cuda.set_per_process_memory_fraction(0.8)
        for i in range(torch.cuda.device_count()):
           print(torch.cuda.get_device_properties(i))
    except:
        print("No GPU")


_CudaDeviceProperties(name='NVIDIA GeForce RTX 3060', major=8, minor=6, total_memory=12287MB, multi_processor_count=28, uuid=9c6396a2-0e5e-4445-8ae5-e68074710ca5, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=2MB)


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
class Tokenizer(transformers.GPT2Tokenizer):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.pad_token_id = self.eos_token_id
        self._tokenizer.post_processor = tokenizers.processors.TemplateProcessing(
            single=f"{self.bos_token} $A {self.eos_token}",
            pair=f"{self.bos_token} $A {self.bos_token} $B {self.eos_token}",
            special_tokens=[
                (self.bos_token, self.bos_token_id),
                (self.eos_token, self.eos_token_id),
            ],
        )
        


In [6]:
tokenizer = Tokenizer.from_pretrained('gpt2')

In [7]:
DATASET_PATH = Path("./dataset")

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

In [8]:
def get_total_sentences():
    en_sentences = pd.read_csv(DATASET_PATH / "eng_sentences.tsv", sep="\t", header=None, names=["en_id", "lang", "sentence"]).drop(columns=["lang"])
    ru_sentences = pd.read_csv(DATASET_PATH / "rus_sentences.tsv", sep="\t", header=None, names=["ru_id", "lang", "sentence"]).drop(columns=["lang"])
    links = pd.read_csv(DATASET_PATH / "links.tsv", sep="\t", header=None, names=["en_id", "ru_id"])
    return (
        links
        .merge(en_sentences, on="en_id", how="inner")
        .merge(ru_sentences, on="ru_id", how="inner")
        .drop(columns=["en_id", "ru_id"])
        .rename(columns={"sentence_x": "en", "sentence_y": "ru"})
    )

In [9]:
total_sentences = get_total_sentences()
total_sentences.head()

,en,ru
0,Let's try something.,Давайте что-нибудь попробуем!
1,Let's try something.,Давайте что-нибудь попробуем.
2,Let's try something.,Давай кое-что попробуем.
3,I have to go to sleep.,Мне пора идти спать.
4,I have to go to sleep.,Мне нужно идти спать.


In [10]:
len(total_sentences), len(total_sentences.groupby("en")), len(total_sentences.groupby("ru"))

(829668, 544615, 710900)

In [11]:
_i = -2
_en = total_sentences.iloc[_i].en
_ru = total_sentences.iloc[_i].ru
(
    _en, len(_en), len(tokenizer.encode(_en, add_special_tokens=False)), len(tokenizer.encode(_en)),
    _ru, len(_ru), len(tokenizer.encode(_ru, add_special_tokens=False)), len(tokenizer.encode(_ru))
)

("The old woman was evicted from an apartment she'd lived her whole life in.",
 74,
 17,
 19,
 'Пожилую женщину выселили из квартиры, в которой она прожила всю жизнь.',
 70,
 76,
 78)

# Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [12]:
class Dataset(keras.utils.PyDataset):

    def __init__(
            self,
            X: list[list[int]],
            y: list[list[int]],
            batch_size: int,
            seed: int | None = None, 
            shuffle: bool = False, 
            **kwargs
    ):
        super().__init__(**kwargs)
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.seed = seed
        self.rng = np.random.default_rng(seed)

        self.X = X
        self.y = y

        self.image_index = np.arange(len(self.X))

        self.on_epoch_end()

    def __len__(self):
        return (len(self.image_index) + self.batch_size - 1) // self.batch_size

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.image_index)
    
    def __getitem__(self, index: int):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch = self.image_index[start:end]

        _X = [self.X[i] for i in batch]
        max_len_X = max(map(len, _X))
        _X = keras.utils.pad_sequences(_X, maxlen=max_len_X, padding='post', truncating='post', value=tokenizer.eos_token_id)
        
        _y = [self.y[i] for i in batch]
        max_len_y = max(map(len, _y))
        
        _mask = keras.utils.pad_sequences(_y, maxlen=max_len_y, padding='post', truncating='post', value=-1)
        _mask = np.where(_mask == -1, 0, 1)
        
        _y = keras.utils.pad_sequences(_y, maxlen=max_len_y, padding='post', truncating='post', value=tokenizer.eos_token_id)

        _y_pred = _y[:, 1:]
        _y_pred = keras.utils.pad_sequences(_y_pred, maxlen=max_len_y, padding='post', truncating='post', value=tokenizer.eos_token_id)


        return (_X, _y), _y_pred, _mask


In [15]:
def get_datasets(df, batch_size: int=32, seed: int=None) -> tuple[Dataset, Dataset]:
    en_encoded = tokenizer.encode(df.en.to_list())
    ru_encoded = tokenizer.encode(df.ru.to_list())
    
    gss = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(df, groups=df.en))

    _train = Dataset([ru_encoded[i] for i in train_idx], [en_encoded[i] for i in train_idx], batch_size=batch_size, seed=seed, shuffle=True)
    _test = Dataset([ru_encoded[i] for i in test_idx], [en_encoded[i] for i in test_idx], batch_size=batch_size, seed=seed, shuffle=False)

    return _train, _test

In [16]:
train, test = get_datasets(total_sentences, batch_size=128, seed=42)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [25]:
def get_name(prefix: str | None = None, suffix: str | None = None, separator: str = '_') -> str | None:
    return prefix and prefix + separator + suffix or suffix or None

In [26]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    attention: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> keras.Model:
    '''Creates a model with RNN architecture for sequence to sequence classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The model'''

    def get_rnn(*, cell_units: int, name_suffix: str, return_sequences: bool, return_state: bool):
        cells = [cell_type(cell_units) for _ in range(n_stacks)]
        stacked_cells = keras.layers.StackedRNNCells(cells)
        return keras.layers.RNN(
            stacked_cells,
            return_sequences=return_sequences,
            return_state=return_state,
            name=get_name("rnn_cell", name_suffix),
        )
    
    inputs_enc = x = keras.layers.Input((None,), dtype='int32', name='input_enc')
    x = keras.layers.Embedding(n_tokens, units, name='embedding_enc')(x)

    rnn_enc = get_rnn(
        cell_units=units,
        name_suffix="enc",
        return_sequences=True,
        return_state=True,
    )

    if bidirectional:
        rnn_enc = keras.layers.Bidirectional(
            rnn_enc, 
            name="bidirectional_enc"
        )

    enc_seq, *enc_states = rnn_enc(x)

    if bidirectional:
        states_per_dir = len(enc_states) // (2 * n_stacks)
        concatenated_states = []
        
        for i in range(n_stacks):
            for j in range(states_per_dir):
                fwd_idx = i * 2 * states_per_dir + j
                bwd_idx = fwd_idx + states_per_dir
                
                fwd_state = enc_states[fwd_idx]
                bwd_state = enc_states[bwd_idx]
                
                concat_state = keras.layers.Concatenate(name=f"concat_state_{i}_{j}")([fwd_state, bwd_state])
                concatenated_states.append(concat_state)
                
        enc_states = concatenated_states

    
    inputs_dec = x = keras.layers.Input((None,), dtype='int32', name='input_dec')
    x = keras.layers.Embedding(n_labels, units, name='embedding_dec')(x)

    rnn_dec = get_rnn(
        cell_units=units * (2 if bidirectional else 1),
        name_suffix="dec",
        return_sequences=True,
        return_state=False,
    )

    dec_seq = x = rnn_dec(x, initial_state=enc_states)

    if attention:
        ctx = keras.layers.AdditiveAttention(name="additive_attention")([dec_seq, enc_seq])
        x = keras.layers.Concatenate(name="concat_dec_attn")([dec_seq, ctx])
 
    x = keras.layers.Dropout(0.2, name="dropout")(x)
    x = keras.layers.Dense(n_labels, name='output')(x)
    return keras.models.Model(
        inputs=[inputs_enc, inputs_dec], 
        outputs=x, 
        name=name
    )

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

In [1]:
models = [
    get_model(
        units=128,
        n_tokens=len(tokenizer.get_vocab()),
        n_labels=len(tokenizer.get_vocab()),
        n_stacks=1,
        bidirectional=False,
        attention=att,
        name=f"GRU-Att-{att}",
        cell_type=keras.layers.GRUCell
    )
    for att in [False, True]
]

NameError: name 'get_model' is not defined

In [28]:
models[1].summary()

Model: "GRU-Att-True"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_enc           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_enc       │ (None, None, 128) │  6,432,896 │ input_enc[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_dec           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_enc   │ [(None, None,     │    396,288 │ embedding_enc[0]… │
│ (Bidirectional)     │ 256), (None,      │            │                   │
│                     │ 128), (None,      │            │                   │
│                     │ 128), (None,      │            │                   │
│                     │ 128), (None,      │            │                   │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_dec       │ (None, None, 128) │  6,432,896 │ input_dec[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_state_0_0    │ (None, 256)       │          0 │ bidirectional_en… │
│ (Concatenate)       │                   │            │ bidirectional_en… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_state_1_0    │ (None, 256)       │          0 │ bidirectional_en… │
│ (Concatenate)       │                   │            │ bidirectional_en… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_cell_dec (RNN)  │ (None, None, 256) │    691,200 │ embedding_dec[0]… │
│                     │                   │            │ concat_state_0_0… │
│                     │                   │            │ concat_state_1_0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention  │ (None, None, 256) │        256 │ rnn_cell_dec[0][… │
│ (AdditiveAttention) │                   │            │ bidirectional_en… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_dec_attn     │ (None, None, 512) │          0 │ rnn_cell_dec[0][… │
│ (Concatenate)       │                   │            │ additive_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None, 512) │          0 │ concat_dec_attn[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │ 25,781,841 │ dropout[0][0]     │
│                     │ 50257)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,735,377 (151.58 MB)

 Trainable params: 39,735,377 (151.58 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
for model in models:
    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=keras.optimizers.Adam(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(),
        ]
    )

In [30]:
SAVE_MODEL_PATH = Path("./saved")

In [ ]:
for model in models:
    print()
    print(model.name)
    
    model.fit(
        train,
        # validation_data=test,
        epochs=10,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                SAVE_MODEL_PATH / f"{model.name}.keras"
            )
        ]
    )


GRU-Att-False
Epoch 1/10


# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [ ]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...